# RL Agent Training Tutorial

This notebook guides you through training a reinforcement learning agent to learn quantum noise models. We'll start with a simple single-circuit example and gradually build up to training on complex multi-qubit datasets.

## 1. Setup and Imports

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Import RL Noise components
from rlnoise.config import (
    DatasetConfig,
    NoiseConfig,
    GateSpecificNoise,
    GymEnvConfig,
    RewardConfig,
    AgentConfig,
)
from rlnoise.dataset import DatasetGenerator, CircuitDataset
from rlnoise.circuit_encoder import CircuitEncoder
from rlnoise.gym_env import QuantumCircuitEnv
from rlnoise.rl_agent import RLAgent

# Set random seeds for reproducibility
np.random.seed(42)

## 2. Understanding the Training Components

Before we start training, let's understand the key components:

### 2.1 Neural Network Architecture

The agent uses a **CNN Feature Extractor** that processes the sliding window observations:
- **Input**: Sliding window of circuit moments (shape: encoding_dim × n_qubits × kernel_size)
- **Conv2D Layer**: Extracts spatial features from the circuit
- **Flatten + Linear**: Maps features to a fixed dimension
- **Policy & Value Networks**: Separate networks for action selection and value estimation

### 2.2 Training Callback

The callback monitors training progress:
- Periodically evaluates the agent on train/validation sets
- Tracks metrics (reward, trace distance, etc.)
- Saves the best model based on validation performance
- Provides real-time feedback during training

### 2.3 Agent Configuration

The `AgentConfig` controls:
- **features_dim**: Output dimension of feature extractor
- **filter_size**: CNN filter width
- **n_filters**: Number of CNN filters
- **n_steps**: Steps before PPO update
- **batch_size**: Batch size for training
- **learning_rate**: Optimizer learning rate

## 3. Simple Training Example: Single Circuit

Let's start with the simplest possible case - a single 1-qubit circuit with only depolarizing noise.

### 3.1. Create a Simple Dataset

In [3]:
# Create a very simple dataset - 1 circuit, 1 qubit, 4 moments
simple_dataset_config = DatasetConfig(
    n_circuits=1,
    moments=4,
    qubits=1,
    primitive_gates=["rx", "rz"],
    clifford=True
)

# Simple noise model - only depolarizing with a clear value
simple_noise_config = NoiseConfig(
    noise_list=[
        GateSpecificNoise(gate="rx", noise_channel="depolarizing", noise_parameter=0.03),
        GateSpecificNoise(gate="rz", noise_channel="depolarizing", noise_parameter=0.03),
    ]
)

# Generate the dataset
generator = DatasetGenerator(simple_dataset_config, simple_noise_config)
simple_dataset = generator.generate(verbose=True)

print(simple_dataset)

Generating 1 circuits...
Applying noise model...
Encoding circuits...
Dataset generated.

  CircuitDataset
    • Circuits:            1
    • Qubits:              1
    • Moments (depth):     4
    • Encoding dimension:  8
    • Circuit shape:       (4, 1, 8)


### 3.2. Create Environment and Encoder

In [4]:
# Create encoder
simple_encoder = CircuitEncoder(primitive_gates=["rx", "rz"])

# Environment configuration
simple_env_config = GymEnvConfig(
    kernel_size=3,
    action_space_max_value=0.1,  # Max noise per step: 0.1
    val_split=0.0,  # No validation split for single circuit
)

# Reward configuration
simple_reward_config = RewardConfig(
    metric="trace",
    function="inverted_squared",
    alpha=20.0
)

# Create environment
simple_env = QuantumCircuitEnv(
    dataset=simple_dataset,
    encoder=simple_encoder,
    env_config=simple_env_config,
    reward_config=simple_reward_config,
)

print(simple_env)

QuantumCircuitEnv:
  Dataset:
    circuits: 1 (1 train, 0 val)
    qubits: 1
  Encoder:
    primitive_gates: ['rx', 'rz']
  Environment:
    observation_space: (8, 1, 3)
    action_space: (1, 4)
    kernel_size: 3
    action_max: 0.1
    only_depolarizing: False
  Reward:
    metric: trace
    function: inverted_squared
    alpha: 20.0


### 3.3. Configure the Agent

For this simple task, we'll use a small neural network.

In [6]:
# Agent configuration - small network for simple task
simple_agent_config = AgentConfig(
    policy="MlpPolicy",
    features_dim=32,        # Number of features extracted by CNN
    filter_size=2,          # Size of convolutional filters
    n_filters=16,           # Number of filters
    pi_net_arch=[32],       # Simple policy network
    vf_net_arch=[32],       # Simple value network
    n_steps=64,             # Update every 64 steps
    batch_size=16,          # Small batches
    learning_rate=3e-4,
    verbose=1
)

simple_agent = RLAgent(
    env=simple_env,
    agent_config=simple_agent_config
)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


### 3.4. Train the Agent

Now let's create the agent and train it!

In [ ]:
# Train the agent
results = simple_agent.train(
    total_timesteps=2000,      # Short training for simple task
    check_freq=500,             # Evaluate every 500 steps
    save_best=False,            # Don't save for this demo
    progress_bar=True
)

print("\n" + "="*60)
print("Training completed!")
print("="*60)

### 3.5. Analyze Training Results

In [ ]:
# Extract results
timesteps = results["timesteps"]
train_results = results["train_results"]

# Plot training progress
if len(timesteps) > 0:
    train_rewards = [r[0] for r in train_results]
    train_stds = [r[1] for r in train_results]
    
    plt.figure(figsize=(10, 5))
    plt.errorbar(timesteps, train_rewards, yerr=train_stds, marker='o', capsize=5)
    plt.xlabel('Training Steps')
    plt.ylabel('Mean Reward')
    plt.title('Training Progress: Single Circuit')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"Initial reward: {train_rewards[0]:.4f}")
    print(f"Final reward: {train_rewards[-1]:.4f}")
    print(f"Improvement: {train_rewards[-1] - train_rewards[0]:.4f}")
else:
    print("Not enough evaluations to plot. Increase total_timesteps or decrease check_freq.")

### 3.6. Test the Trained Agent

Let's see what noise parameters the agent learned to apply!

In [ ]:
# Reset environment and run an episode with the trained agent
obs, info = simple_env.reset()

print("Running trained agent on the circuit...\n")
print("Expected: depolarizing ≈ 0.3 (scaled by 0.1 = 0.03)")
print("="*60)

actions_taken = []
step = 0
terminated = False

while not terminated:
    # Get action from trained agent
    action = simple_agent.predict(obs, deterministic=True)
    actions_taken.append(action)
    
    # Show what action was taken
    depol_action = action[0, 3]  # Index 3 is depolarizing
    print(f"Step {step}: Depolarizing action = {depol_action:.4f} → noise = {depol_action * 0.1:.4f}")
    
    # Take step
    obs, reward, terminated, truncated, info = simple_env.step(action)
    step += 1

print("="*60)

# Calculate mean action
mean_depol_action = np.mean([a[0, 3] for a in actions_taken])
print(f"\nMean depolarizing action: {mean_depol_action:.4f}")
print(f"Corresponding noise: {mean_depol_action * 0.1:.4f}")
print(f"Target noise: 0.0300")
print(f"Error: {abs(mean_depol_action * 0.1 - 0.03):.4f}")

### 3.7. Compare with Random Agent

In [ ]:
# Evaluate random agent
print("Evaluating Random Agent (baseline):")
random_rewards = []

for _ in range(10):
    obs, _ = simple_env.reset()
    terminated = False
    while not terminated:
        action = simple_env.action_space.sample()  # Random action
        obs, reward, terminated, truncated, info = simple_env.step(action)
    random_rewards.append(reward)

random_mean = np.mean(random_rewards)
random_std = np.std(random_rewards)

# Evaluate trained agent
print("\nEvaluating Trained Agent:")
trained_metrics = simple_agent.evaluate(n_episodes=10, deterministic=True)

print("\n" + "="*60)
print("COMPARISON")
print("="*60)
print(f"Random Agent:  {random_mean:.4f} ± {random_std:.4f}")
print(f"Trained Agent: {trained_metrics['mean_reward']:.4f} ± {trained_metrics['std_reward']:.4f}")
print(f"Improvement: {trained_metrics['mean_reward'] - random_mean:.4f}")
print("="*60)

## 4. Training with Multiple Circuits

Now let's scale up to multiple circuits to see how the agent generalizes.

### 4.1. Create Multi-Circuit Dataset

In [ ]:
# Create dataset with multiple circuits
multi_dataset_config = DatasetConfig(
    n_circuits=10,          # 10 circuits
    moments=6,              # 6 moments each
    qubits=1,
    primitive_gates=["rx", "rz"],
    clifford=True
)

# Same noise model
multi_noise_config = NoiseConfig(
    noise_list=[
        GateSpecificNoise(gate="rx", noise_channel="depolarizing", noise_parameter=0.03),
        GateSpecificNoise(gate="rz", noise_channel="depolarizing", noise_parameter=0.03),
    ]
)

print("Generating multi-circuit dataset...")
generator = DatasetGenerator(multi_dataset_config, multi_noise_config)
multi_dataset = generator.generate(verbose=False)

print(f"\nDataset: {len(multi_dataset)} circuits")

### 4.2. Setup Environment with Validation Split

In [ ]:
# Environment with validation split
multi_env_config = GymEnvConfig(
    kernel_size=3,
    action_space_max_value=0.1,
    val_split=0.2,  # 20% for validation
)

multi_env = QuantumCircuitEnv(
    dataset=multi_dataset,
    encoder=simple_encoder,
    env_config=multi_env_config,
    reward_config=simple_reward_config,
)

print(f"Training circuits: {multi_env.n_circuits_train}")
print(f"Validation circuits: {multi_env.n_circuits - multi_env.n_circuits_train}")

### 4.3. Train with Callback Monitoring

In [ ]:
# Create agent with slightly larger network
multi_agent_config = AgentConfig(
    features_dim=64,
    filter_size=2,
    n_filters=24,
    pi_net_arch=[32, 32],
    vf_net_arch=[32, 32],
    n_steps=128,
    batch_size=32,
    learning_rate=3e-4,
    verbose=1
)

multi_agent = RLAgent(
    env=multi_env,
    agent_config=multi_agent_config
)

# Train with callback
print("Training on multiple circuits...")
multi_results = multi_agent.train(
    total_timesteps=5000,
    check_freq=500,
    save_best=False,
    progress_bar=True
)

print("\nTraining completed!")

### 4.4. Analyze Multi-Circuit Training

In [ ]:
# Plot training and validation curves
timesteps = multi_results["timesteps"]
train_results = multi_results["train_results"]
val_results = multi_results["eval_results"]

if len(timesteps) > 0:
    train_rewards = [r[0] for r in train_results]
    val_rewards = [r[0] for r in val_results]
    
    plt.figure(figsize=(12, 5))
    
    # Rewards
    plt.subplot(1, 2, 1)
    plt.plot(timesteps, train_rewards, 'o-', label='Training', linewidth=2)
    plt.plot(timesteps, val_rewards, 's-', label='Validation', linewidth=2)
    plt.xlabel('Training Steps')
    plt.ylabel('Mean Reward')
    plt.title('Training Progress')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Learning curve
    plt.subplot(1, 2, 2)
    improvement_train = np.array(train_rewards) - train_rewards[0]
    improvement_val = np.array(val_rewards) - val_rewards[0]
    plt.plot(timesteps, improvement_train, 'o-', label='Train Improvement', linewidth=2)
    plt.plot(timesteps, improvement_val, 's-', label='Val Improvement', linewidth=2)
    plt.xlabel('Training Steps')
    plt.ylabel('Reward Improvement')
    plt.title('Learning Progress')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.axhline(y=0, color='k', linestyle='--', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Training - Initial: {train_rewards[0]:.4f}, Final: {train_rewards[-1]:.4f}")
    print(f"Validation - Initial: {val_rewards[0]:.4f}, Final: {val_rewards[-1]:.4f}")

## 5. Advanced: 3-Qubit Training

Now for the final challenge - training on 3-qubit circuits with multiple noise types!

### 5.1. Load 3-Qubit Dataset

We'll use a pre-generated 3-qubit dataset from the examples folder.

In [ ]:
# Load the 3-qubit dataset
dataset_path = Path("example_datasets/dataset_3q.npz")

if dataset_path.exists():
    dataset_3q = CircuitDataset.load(str(dataset_path))
    print("Loaded 3-qubit dataset:")
    print(dataset_3q)
else:
    print("3-qubit dataset not found. Generating it...")
    # Generate 3-qubit dataset
    dataset_3q_config = DatasetConfig(
        n_circuits=20,
        moments=8,
        qubits=3,
        primitive_gates=["rx", "rz", "cz"],
        clifford=True
    )
    
    noise_3q_config = NoiseConfig(
        noise_list=[
            GateSpecificNoise(gate="rx", noise_channel="depolarizing", noise_parameter=0.02),
            GateSpecificNoise(gate="rz", noise_channel="depolarizing", noise_parameter=0.02),
            GateSpecificNoise(gate="cz", noise_channel="depolarizing", noise_parameter=0.03),
            GateSpecificNoise(gate="rx", noise_channel="damping", noise_parameter=0.01),
            GateSpecificNoise(gate="rz", noise_channel="damping", noise_parameter=0.01),
        ]
    )
    
    generator_3q = DatasetGenerator(dataset_3q_config, noise_3q_config)
    dataset_3q = generator_3q.generate(verbose=False)
    print(dataset_3q)

### 5.2. Setup 3-Qubit Environment

In [ ]:
# Create encoder for 3-qubit circuits
encoder_3q = CircuitEncoder(primitive_gates=["rx", "rz", "cz"])

# Environment configuration
env_config_3q = GymEnvConfig(
    kernel_size=3,
    action_space_max_value=0.1,
    val_split=0.2,
    enable_only_depolarizing=False,  # Allow all noise types
)

# Reward configuration
reward_config_3q = RewardConfig(
    metric="trace",
    function="inverted_squared",
    alpha=20.0
)

# Create environment
env_3q = QuantumCircuitEnv(
    dataset=dataset_3q,
    encoder=encoder_3q,
    env_config=env_config_3q,
    reward_config=reward_config_3q,
)

print(env_3q)

### 5.3. Configure Agent for 3-Qubit Circuits

For 3-qubit circuits, we need a larger network to handle the complexity.

In [ ]:
# Agent configuration - larger network for 3-qubit circuits
agent_config_3q = AgentConfig(
    policy="MlpPolicy",
    features_dim=128,       # Larger feature dimension
    filter_size=3,          # Match kernel size
    n_filters=64,           # More filters for complexity
    pi_net_arch=[128, 64],  # Deeper policy network
    vf_net_arch=[128, 64],  # Deeper value network
    n_steps=512,            # More steps before update
    batch_size=64,          # Larger batches
    learning_rate=3e-4,
    gamma=0.99,
    verbose=1
)

print("3-Qubit Agent Configuration:")
print(f"  Features dimension: {agent_config_3q.features_dim}")
print(f"  CNN filters: {agent_config_3q.n_filters}")
print(f"  Policy network: {agent_config_3q.pi_net_arch}")
print(f"  Value network: {agent_config_3q.vf_net_arch}")
print(f"  Batch size: {agent_config_3q.batch_size}")

### 5.4. Train 3-Qubit Agent

This will take longer due to the increased complexity.

In [ ]:
# Create and train the 3-qubit agent
agent_3q = RLAgent(
    env=env_3q,
    agent_config=agent_config_3q
)

print("Training 3-qubit agent...")
print("This may take a few minutes...\n")

# Train with model saving
results_3q = agent_3q.train(
    total_timesteps=10000,
    check_freq=1000,
    save_best=False,
    progress_bar=True
)

print("\n" + "="*60)
print("3-Qubit Training Completed!")
print("="*60)

### 5.5. Visualize 3-Qubit Training Results

In [ ]:
# Extract and plot results
timesteps_3q = results_3q["timesteps"]
train_results_3q = results_3q["train_results"]
val_results_3q = results_3q["eval_results"]

if len(timesteps_3q) > 0:
    train_rewards_3q = [r[0] for r in train_results_3q]
    train_stds_3q = [r[1] for r in train_results_3q]
    val_rewards_3q = [r[0] for r in val_results_3q]
    val_stds_3q = [r[1] for r in val_results_3q]
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Training curves
    ax = axes[0]
    ax.errorbar(timesteps_3q, train_rewards_3q, yerr=train_stds_3q, 
                marker='o', label='Training', capsize=5, linewidth=2)
    ax.errorbar(timesteps_3q, val_rewards_3q, yerr=val_stds_3q, 
                marker='s', label='Validation', capsize=5, linewidth=2)
    ax.set_xlabel('Training Steps', fontsize=12)
    ax.set_ylabel('Mean Reward', fontsize=12)
    ax.set_title('3-Qubit Training Progress', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # Learning rate
    ax = axes[1]
    improvement_train = np.array(train_rewards_3q) - train_rewards_3q[0]
    improvement_val = np.array(val_rewards_3q) - val_rewards_3q[0]
    ax.plot(timesteps_3q, improvement_train, 'o-', label='Train', linewidth=2)
    ax.plot(timesteps_3q, improvement_val, 's-', label='Validation', linewidth=2)
    ax.axhline(y=0, color='k', linestyle='--', alpha=0.3)
    ax.set_xlabel('Training Steps', fontsize=12)
    ax.set_ylabel('Reward Improvement', fontsize=12)
    ax.set_title('Learning Progress', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # Final comparison
    ax = axes[2]
    categories = ['Train\nInitial', 'Train\nFinal', 'Val\nInitial', 'Val\nFinal']
    values = [train_rewards_3q[0], train_rewards_3q[-1], 
              val_rewards_3q[0], val_rewards_3q[-1]]
    colors = ['lightcoral', 'darkgreen', 'lightblue', 'darkblue']
    bars = ax.bar(categories, values, color=colors, edgecolor='black', linewidth=1.5)
    ax.set_ylabel('Mean Reward', fontsize=12)
    ax.set_title('Performance Summary', fontsize=14, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for bar, value in zip(bars, values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{value:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print("\nTraining Summary:")
    print("="*60)
    print(f"Training Set:")
    print(f"  Initial reward: {train_rewards_3q[0]:.4f}")
    print(f"  Final reward:   {train_rewards_3q[-1]:.4f}")
    print(f"  Improvement:    {train_rewards_3q[-1] - train_rewards_3q[0]:.4f}")
    print(f"\nValidation Set:")
    print(f"  Initial reward: {val_rewards_3q[0]:.4f}")
    print(f"  Final reward:   {val_rewards_3q[-1]:.4f}")
    print(f"  Improvement:    {val_rewards_3q[-1] - val_rewards_3q[0]:.4f}")
    print("="*60)

### 5.6. Apply Trained Agent to Circuits

In [ ]:
# Get a circuit from the validation set
val_circuit_idx = env_3q.n_circuits_train
circuit_array = dataset_3q.circuits[val_circuit_idx]

print(f"Applying trained agent to validation circuit {val_circuit_idx}")
print(f"Circuit shape: {circuit_array.shape}")

# Apply trained agent
noisy_circuit_trained = agent_3q.apply_to_circuit(circuit_array, return_qibo=True)

# Apply random agent for comparison
obs, _ = env_3q.reset(options={"circuit_idx": val_circuit_idx})
terminated = False
while not terminated:
    action = env_3q.action_space.sample()
    obs, reward_random, terminated, truncated, info = env_3q.step(action)

noisy_circuit_random = encoder_3q.array_to_circuit(env_3q.current_circuit)

# Get reward for trained agent
obs, _ = env_3q.reset(options={"circuit_idx": val_circuit_idx})
terminated = False
while not terminated:
    action = agent_3q.predict(obs, deterministic=True)
    obs, reward_trained, terminated, truncated, info = env_3q.step(action)

print("\n" + "="*70)
print("TRAINED AGENT - Circuit with Learned Noise")
print("="*70)
print(noisy_circuit_trained.draw())
print(f"\nReward: {reward_trained:.4f}")

print("\n" + "="*70)
print("RANDOM AGENT - Circuit with Random Noise")
print("="*70)
print(noisy_circuit_random.draw())
print(f"\nReward: {reward_random:.4f}")

print("\n" + "="*70)
print(f"Trained agent achieves {reward_trained:.4f} vs Random {reward_random:.4f}")
print(f"Improvement: {reward_trained - reward_random:.4f}")
print("="*70)

### 5.7. Comprehensive Evaluation

In [ ]:
# Evaluate on validation set
print("Evaluating trained agent on validation set...")
trained_metrics = agent_3q.evaluate(n_episodes=env_3q.n_circuits - env_3q.n_circuits_train, deterministic=True)

# Evaluate random agent
print("Evaluating random agent on validation set...")
random_rewards_3q = []

for i in range(env_3q.n_circuits_train, env_3q.n_circuits):
    obs, _ = env_3q.reset(options={"circuit_idx": i})
    terminated = False
    while not terminated:
        action = env_3q.action_space.sample()
        obs, reward, terminated, truncated, info = env_3q.step(action)
    random_rewards_3q.append(reward)

random_metrics = {
    "mean_reward": np.mean(random_rewards_3q),
    "std_reward": np.std(random_rewards_3q),
    "min_reward": np.min(random_rewards_3q),
    "max_reward": np.max(random_rewards_3q),
}

# Display comparison
print("\n" + "="*70)
print("FINAL EVALUATION ON VALIDATION SET")
print("="*70)
print(f"Random Agent:")
print(f"  Mean Reward: {random_metrics['mean_reward']:.4f} ± {random_metrics['std_reward']:.4f}")
print(f"  Min Reward:  {random_metrics['min_reward']:.4f}")
print(f"  Max Reward:  {random_metrics['max_reward']:.4f}")
print()
print(f"Trained Agent:")
print(f"  Mean Reward: {trained_metrics['mean_reward']:.4f} ± {trained_metrics['std_reward']:.4f}")
print(f"  Min Reward:  {trained_metrics['min_reward']:.4f}")
print(f"  Max Reward:  {trained_metrics['max_reward']:.4f}")
print()
print(f"Performance Improvement:")
print(f"  Absolute: +{trained_metrics['mean_reward'] - random_metrics['mean_reward']:.4f}")
print(f"  Relative: {(trained_metrics['mean_reward'] / random_metrics['mean_reward'] - 1) * 100:.2f}%")
print("="*70)

# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot comparison
ax = axes[0]
data_to_plot = [random_rewards_3q, 
                [trained_metrics['mean_reward']] * len(random_rewards_3q)]  # Approximate
bp = ax.boxplot(data_to_plot, labels=['Random', 'Trained'], patch_artist=True)
bp['boxes'][0].set_facecolor('lightcoral')
bp['boxes'][1].set_facecolor('lightgreen')
ax.set_ylabel('Reward', fontsize=12)
ax.set_title('Reward Distribution Comparison', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Bar comparison
ax = axes[1]
agents = ['Random', 'Trained']
means = [random_metrics['mean_reward'], trained_metrics['mean_reward']]
stds = [random_metrics['std_reward'], trained_metrics['std_reward']]
colors = ['lightcoral', 'lightgreen']
bars = ax.bar(agents, means, yerr=stds, color=colors, capsize=10, 
              edgecolor='black', linewidth=2)
ax.set_ylabel('Mean Reward', fontsize=12)
ax.set_title('Agent Performance Comparison', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

for bar, mean in zip(bars, means):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{mean:.3f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Summary and Key Takeaways

### What We Learned:

1. **Start Simple**: Begin with single circuits and simple noise models to verify training works
2. **Scale Gradually**: Move from 1 circuit → multiple circuits → multi-qubit circuits
3. **Monitor Training**: Use callbacks to track progress and detect issues early
4. **Configure Carefully**: Larger/more complex circuits need larger networks and more training steps
5. **Validate Performance**: Always compare trained agents against baselines (random actions)

### Training Tips:

- **Hyperparameters**: Start with small networks and increase gradually
- **Check Freq**: Evaluate frequently early on to catch problems
- **Learning Rate**: 3e-4 is a good default, adjust if training is unstable
- **Batch Size**: Larger batches = more stable but slower training
- **N Steps**: Balance between sample efficiency and update frequency

### Next Steps:

- Save trained models for later use: `agent.save("path/to/model")`
- Load models: `agent = RLAgent(env, config, model_path="path/to/model")`
- Apply to hardware calibration data
- Experiment with different reward functions
- Try different network architectures

## Congratulations! 🎉

You've successfully trained RL agents to learn quantum noise models, from simple single-circuit examples to complex 3-qubit circuits. The trained agents can now predict appropriate noise parameters for quantum circuits, which is crucial for accurate quantum simulation and error mitigation.